# Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [3]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

### VectorDB Instance

In [5]:
openai_api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")

In [6]:
from lib.vector_db import VectorStoreManager, VectorStore
manager = VectorStoreManager(
    openai_api_key=openai_api_key,
    persistent_directory="chromadb"
)

In [7]:
manager

VectorStoreManager(mode=persistent):<chromadb.api.client.Client object at 0x739e9af87e00>

### Collection

In [ ]:
"""
embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

collection = chroma_client.create_collection(
   name="udaplay",
   embedding_function=embedding_fn
)

"""

In [8]:
store = manager.get_or_create_store("udaplay")
collection = store._collection

### Add documents

In [9]:
from lib.documents import Document

data_dir = "games"
documents = []

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]
    documents.append(
        Document(
            id=doc_id,
            content=content,
            metadata=game
        )
    )

store.add(documents)

Sanity Check

In [10]:
all_docs = store.get()
print(f"Documents in collection: {len(all_docs['ids'])}")

Documents in collection: 15


Semantic Search

In [11]:
def show_results(results):
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]
    for doc, meta, dist in zip(docs, metas, dists):
        similarity = 1 - dist
        print(f"({similarity:.3f}) [{meta.get('Platform')}] {meta.get('Name')} ({meta.get('YearOfRelease')})")
        print(f"    {doc}")
    print()

queries = [
    "racing simulator with realistic cars",
    "open world action adventure game",
    "a platformer starring an Italian plumber",
]

for q in queries:
    print(f"=== Query: {q!r} ===")
    # Passing a list explicitly here, even though VectorStore.query()'s type
    # allows a bare str - chromadb's Collection.query() expects
    # query_texts to be a list, so a single string is risky to rely on.
    results = store.query(query_texts=[q], n_results=3)
    show_results(results)

=== Query: 'racing simulator with realistic cars' ===
(0.882) [PlayStation 1] Gran Turismo (1997)
    [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.
(0.871) [PlayStation 3] Gran Turismo 5 (2010)
    [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.
(0.801) [Nintendo Switch] Mario Kart 8 Deluxe (2017)
    [Nintendo Switch] Mario Kart 8 Deluxe (2017) - An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.

=== Query: 'open world action adventure game' ===
(0.837) [PlayStation 4] Marvel's Spider-Man (2018)
    [PlayStation 4] Marvel's Spider-Man (2018) - An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.
(0.826) [Xbox One] Minecraft (2014)
    [Xbox One] Minecraft (2014) -

Metadata filtering

In [12]:
results = store.query(
    query_texts=["a fun, family-friendly game"],
    n_results=3,
    where={"Platform": "PlayStation 2"}  # <-- set to a real platform in your dataset
)
show_results(results)

(0.773) [PlayStation 2] Grand Theft Auto: San Andreas (2004)
    [PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.



Reusability check

In [13]:
reconnect_manager = VectorStoreManager(openai_api_key=openai_api_key, persistent_directory="chromadb")
reconnected_store = reconnect_manager.get_store("udaplay")

assert reconnected_store is not None, "Expected an existing 'udaplay' collection on disk - run the Add documents cell first."
print(f"Reconnected store has {len(reconnected_store.get()['ids'])} documents (no re-adding needed)")

Reconnected store has 15 documents (no re-adding needed)
